In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize","1073741824")

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, 3, Finished, Available, Finished, False)

In [2]:
from notebookutils import mssparkutils
adventureWorksPath = "abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/a9647c27-8250-4e02-bdeb-089a9e854410/Files/vbdsqldb"

file_list = mssparkutils.fs.ls(adventureWorksPath)

# Read each file and create a DataFrame
for file_path in file_list:
    print(file_path)
    df = spark.read.format("csv").options(inferSchema="true", header="true").load(path=f"{file_path.path}*")
    # You can process the DataFrame or register it as a table here
    # For example, to create a temporary table:
    df.createOrReplaceTempView(file_path.name.removesuffix('.csv'))

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, 4, Finished, Available, Finished, False)

FileInfo(path=abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/a9647c27-8250-4e02-bdeb-089a9e854410/Files/vbdsqldb/salesltaddress.csv, name=salesltaddress.csv, size=55719)
FileInfo(path=abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/a9647c27-8250-4e02-bdeb-089a9e854410/Files/vbdsqldb/salesltcustomer.csv, name=salesltcustomer.csv, size=149308)
FileInfo(path=abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/a9647c27-8250-4e02-bdeb-089a9e854410/Files/vbdsqldb/salesltcustomeraddress.csv, name=salesltcustomeraddress.csv, size=14677)
FileInfo(path=abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/a9647c27-8250-4e02-bdeb-089a9e854410/Files/vbdsqldb/salesltproduct.csv, name=salesltproduct.csv, size=9851)
FileInfo(path=abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/a9647c27-8250-4e02-bdeb-089a9e854410/Files/vbdsqldb/salesltproductcategory.csv, name=s

In [3]:
%%sql
SHOW VIEWS

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, 5, Finished, Available, Finished, False)

Error: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.

In [ ]:
%%sql
SELECT * FROM salesltaddress LIMIT 100

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
views = spark.catalog.listTables()
display(views)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

# Read Temp Views, clean the data, load into Spark Dataframes
- Filter Rows
- Rename Columns
- Drop Columns

In [ ]:
df_salesltsalesorderdetail = spark.sql("SELECT SalesOrderID, OrderQty, ProductID,UnitPrice, UnitPriceDIscount, LineTotal FROM salesltsalesorderdetail")
display(df_salesltsalesorderdetail)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
df_salesltsalesorderheader = spark.sql("SELECT SalesOrderID, RevisionNumber, OrderDate, DueDate, ShipDate, Status, OnlineOrderFlag, SalesOrderNumber, PurchaseOrderNumber, AccountNumber, CustomerID, ShipToAddressID, BillToAddressID, ShipMethod, SubTotal, TaxAmt, Freight, TotalDue FROM salesltsalesorderheader")
display(df_salesltsalesorderheader)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
# Randomizing the dates in the OrderDate column since our toy AdventureWorks LT dataset only has one distinct order date.
from pyspark.sql.functions import rand, col, expr
df_salesltsalesorderheader = df_salesltsalesorderheader.drop("OrderDate").withColumn("OrderDate", expr("date_add(current_date()-1000, CAST(rand() * 365 AS INT))"))
display(df_salesltsalesorderheader)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
df_salesltcustomer = spark.sql("SELECT CustomerID, Title, FirstName, MiddleName, LastName, Suffix, CompanyName, EmailAddress, Phone  FROM salesltcustomer")
display(df_salesltcustomer)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
df_salesltcustomeraddress = spark.sql("SELECT CustomerID, AddressID, AddressType FROM salesltcustomeraddress")
display(df_salesltcustomeraddress)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
df_salesltproduct = spark.sql("SELECT ProductID, Name, ProductNumber, Color, StandardCost, ListPrice, Size, Weight, ProductCategoryID FROM salesltproduct")
display(df_salesltproduct)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
df_salesltproductcategory = spark.sql("SELECT ProductCategoryID, ParentProductCategoryID, Name FROM salesltproductcategory")
display(df_salesltproductcategory)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

# Write Data Frames into Silver Lakehouse as Delta Tables

In [ ]:
basePathSilverLakeHouse = "abfss://c46e3841-d23d-4827-b8dd-adc1d8432c46@onelake.dfs.fabric.microsoft.com/519d7209-4d05-47ab-b96a-55edb8b87e9b/Tables"
tableName="salesOrderHeader"
df_salesltsalesorderheader.write.mode("overwrite").format("delta").save(basePathSilverLakeHouse + '//' + tableName)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
tableName="salesOrderDetail"
df_salesltsalesorderdetail.write.mode("overwrite").format("delta").save(basePathSilverLakeHouse + '//' + tableName)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
tableName="salesCustomer"
df_salesltcustomer.write.mode("overwrite").format("delta").save(basePathSilverLakeHouse + '//' + tableName)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
tableName="salesCustomerAddress"
df_salesltcustomeraddress.write.mode("overwrite").format("delta").save(basePathSilverLakeHouse + '//' + tableName)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
tableName="salesProduct"
df_salesltproduct.write.mode("overwrite").format("delta").save(basePathSilverLakeHouse + '//' + tableName)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)

In [ ]:
tableName="salesProductCategory"
df_salesltproductcategory.write.mode("overwrite").format("delta").save(basePathSilverLakeHouse + '//' + tableName)

StatementMeta(, 7c5ded9b-0352-4b71-a327-1857ab85118e, -1, Cancelled, , Cancelled, True)